# Single-Echelon Safety Stock Playground

Use this notebook to explore how demand variability, lead-time variability, and service targets change safety stock and reorder points.

What you can do in class:
- Run a baseline cycle service level example
- Run a fill-rate example
- Change one input at a time and discuss impact
- Perform a quick service-level sensitivity scan

In [3]:
from dataclasses import dataclass, asdict
from pathlib import Path
from statistics import NormalDist
from typing import Literal, Optional
import math
import sys


_loaded_from = None

# 1) Try normal import first (works if package is installed in kernel env)
try:
    from safety_stock.calculator import SafetyStockCalculator
    from safety_stock.models import DemandLeadTimeProfile, InputScenario, ServiceTarget
    _loaded_from = "installed package"
except Exception:
    # 2) Try local source import from current working tree
    repo_root = None
    start = Path.cwd()
    for root in [start, *start.parents]:
        src_pkg = root / "src" / "safety_stock"
        if (src_pkg / "calculator.py").exists() and (src_pkg / "models.py").exists():
            repo_root = root
            break

    if repo_root is not None:
        src_dir = repo_root / "src"
        if str(src_dir) not in sys.path:
            sys.path.insert(0, str(src_dir))

        for name in list(sys.modules):
            if name == "safety_stock" or name.startswith("safety_stock."):
                del sys.modules[name]

        from safety_stock.calculator import SafetyStockCalculator
        from safety_stock.models import DemandLeadTimeProfile, InputScenario, ServiceTarget
        _loaded_from = f"local source at {repo_root}"
    else:
        # 3) Fallback: inline implementation for virtual GitHub FS environments
        ServiceTargetKind = Literal["cycle_service_level", "fill_rate"]

        @dataclass(frozen=True)
        class DemandLeadTimeProfile:
            mean_demand_per_period: float
            stdev_demand_per_period: float
            mean_lead_time_periods: float
            stdev_lead_time_periods: float = 0.0
            review_period_periods: float = 0.0

            def validate(self) -> None:
                if self.mean_demand_per_period <= 0:
                    raise ValueError("mean_demand_per_period must be > 0")
                if self.stdev_demand_per_period < 0:
                    raise ValueError("stdev_demand_per_period must be >= 0")
                if self.mean_lead_time_periods <= 0:
                    raise ValueError("mean_lead_time_periods must be > 0")
                if self.stdev_lead_time_periods < 0:
                    raise ValueError("stdev_lead_time_periods must be >= 0")
                if self.review_period_periods < 0:
                    raise ValueError("review_period_periods must be >= 0")

        @dataclass(frozen=True)
        class ServiceTarget:
            kind: ServiceTargetKind
            value: float
            order_quantity: Optional[float] = None

            def validate(self) -> None:
                if not (0 < self.value < 1):
                    raise ValueError("Service target value must be between 0 and 1")
                if self.kind == "fill_rate":
                    if self.order_quantity is None or self.order_quantity <= 0:
                        raise ValueError("fill_rate target requires order_quantity > 0")

        @dataclass(frozen=True)
        class InputScenario:
            sku: str
            profile: DemandLeadTimeProfile
            target: ServiceTarget

            def validate(self) -> None:
                if not self.sku.strip():
                    raise ValueError("sku must not be empty")
                self.profile.validate()
                self.target.validate()

        @dataclass(frozen=True)
        class CalculationResult:
            sku: str
            target_kind: ServiceTargetKind
            target_value: float
            z_value: float
            expected_demand_in_protection_period: float
            stdev_demand_in_protection_period: float
            safety_stock: float
            reorder_point: float
            expected_shortage_per_cycle: Optional[float] = None

        class SafetyStockCalculator:
            _normal = NormalDist(mu=0.0, sigma=1.0)

            def calculate(self, scenario: InputScenario) -> CalculationResult:
                scenario.validate()

                mean_pp = self._protection_period_mean(scenario)
                stdev_pp = self._protection_period_stdev(scenario)
                target = scenario.target

                if stdev_pp == 0:
                    z_value = 0.0
                    safety_stock = 0.0
                    expected_shortage = 0.0 if target.kind == "fill_rate" else None
                elif target.kind == "cycle_service_level":
                    z_value = self._normal.inv_cdf(target.value)
                    safety_stock = z_value * stdev_pp
                    expected_shortage = None
                else:
                    order_qty = float(target.order_quantity)
                    z_value = self._solve_z_for_fill_rate(target.value, order_qty, stdev_pp)
                    safety_stock = z_value * stdev_pp
                    expected_shortage = self._expected_shortage_per_cycle(z_value, stdev_pp)

                reorder_point = mean_pp + safety_stock

                return CalculationResult(
                    sku=scenario.sku,
                    target_kind=target.kind,
                    target_value=target.value,
                    z_value=z_value,
                    expected_demand_in_protection_period=mean_pp,
                    stdev_demand_in_protection_period=stdev_pp,
                    safety_stock=safety_stock,
                    reorder_point=reorder_point,
                    expected_shortage_per_cycle=expected_shortage,
                )

            @staticmethod
            def _protection_period_mean(scenario: InputScenario) -> float:
                p = scenario.profile
                return p.mean_demand_per_period * (p.mean_lead_time_periods + p.review_period_periods)

            @staticmethod
            def _protection_period_stdev(scenario: InputScenario) -> float:
                p = scenario.profile
                protection_window = p.mean_lead_time_periods + p.review_period_periods
                demand_component = protection_window * (p.stdev_demand_per_period ** 2)
                lead_time_component = (p.mean_demand_per_period ** 2) * (p.stdev_lead_time_periods ** 2)
                variance = demand_component + lead_time_component
                return math.sqrt(variance)

            def _solve_z_for_fill_rate(self, fill_rate: float, order_qty: float, sigma_pp: float) -> float:
                target_shortage = (1 - fill_rate) * order_qty
                lo, hi = -4.0, 8.0
                for _ in range(80):
                    mid = 0.5 * (lo + hi)
                    shortage = self._expected_shortage_per_cycle(mid, sigma_pp)
                    if shortage > target_shortage:
                        lo = mid
                    else:
                        hi = mid
                return 0.5 * (lo + hi)

            def _expected_shortage_per_cycle(self, z_value: float, sigma_pp: float) -> float:
                cdf = self._normal.cdf(z_value)
                pdf = self._normal.pdf(z_value)
                loss = pdf - z_value * (1 - cdf)
                return sigma_pp * loss

        _loaded_from = "inline fallback implementation"


calculator = SafetyStockCalculator()
print(f"Calculator ready ({_loaded_from}).")

Calculator ready (inline fallback implementation).


In [4]:
def show_result(result):
    data = asdict(result)
    print(f"SKU: {data['sku']}")
    print(f"Target type: {data['target_kind']}")
    print(f"Target value: {data['target_value']:.3f}")
    print(f"z-value: {data['z_value']:.3f}")
    print(f"Mean demand in protection period: {data['expected_demand_in_protection_period']:.2f}")
    print(f"Std dev in protection period: {data['stdev_demand_in_protection_period']:.2f}")
    print(f"Safety stock: {data['safety_stock']:.2f}")
    print(f"Reorder point: {data['reorder_point']:.2f}")
    if data['expected_shortage_per_cycle'] is not None:
        print(f"Expected shortage per cycle: {data['expected_shortage_per_cycle']:.2f}")

## 1) Cycle Service Level Example

This target controls the probability of no stockout during a replenishment cycle.

In [5]:
scenario_csl = InputScenario(
    sku="CLASSROOM-CSL",
    profile=DemandLeadTimeProfile(
        mean_demand_per_period=120,
        stdev_demand_per_period=25,
        mean_lead_time_periods=2.5,
        stdev_lead_time_periods=0.6,
        review_period_periods=0.0,
    ),
    target=ServiceTarget(kind="cycle_service_level", value=0.95),
)

result_csl = calculator.calculate(scenario_csl)
show_result(result_csl)

SKU: CLASSROOM-CSL
Target type: cycle_service_level
Target value: 0.950
z-value: 1.645
Mean demand in protection period: 300.00
Std dev in protection period: 82.14
Safety stock: 135.10
Reorder point: 435.10


## 2) Fill Rate Example

This target controls the fraction of demand immediately fulfilled from stock.

In [6]:
scenario_fill = InputScenario(
    sku="CLASSROOM-FILL",
    profile=DemandLeadTimeProfile(
        mean_demand_per_period=80,
        stdev_demand_per_period=15,
        mean_lead_time_periods=3.0,
        stdev_lead_time_periods=0.8,
        review_period_periods=1.0,
    ),
    target=ServiceTarget(kind="fill_rate", value=0.98, order_quantity=400),
)

result_fill = calculator.calculate(scenario_fill)
show_result(result_fill)

SKU: CLASSROOM-FILL
Target type: fill_rate
Target value: 0.980
z-value: 0.834
Mean demand in protection period: 320.00
Std dev in protection period: 70.68
Safety stock: 58.94
Reorder point: 378.94
Expected shortage per cycle: 8.00


## 3) Sensitivity: Service Level vs Safety Stock

Edit the service level range below and rerun to see how aggressively inventory grows at high targets.

In [7]:
base_profile = DemandLeadTimeProfile(
    mean_demand_per_period=100,
    stdev_demand_per_period=20,
    mean_lead_time_periods=2.0,
    stdev_lead_time_periods=0.5,
)

service_levels = [0.90, 0.92, 0.95, 0.97, 0.98, 0.99]
rows = []
for level in service_levels:
    scenario = InputScenario(
        sku=f"SENS-{level:.2f}",
        profile=base_profile,
        target=ServiceTarget(kind="cycle_service_level", value=level),
    )
    result = calculator.calculate(scenario)
    rows.append((level, result.z_value, result.safety_stock, result.reorder_point))

print(f"{'Service':>8} {'z':>8} {'SafetyStock':>14} {'ReorderPoint':>14}")
for level, z_value, ss, rop in rows:
    print(f"{level:>8.2%} {z_value:>8.3f} {ss:>14.2f} {rop:>14.2f}")

 Service        z    SafetyStock   ReorderPoint
  90.00%    1.282          73.62         273.62
  92.00%    1.405          80.72         280.72
  95.00%    1.645          94.49         294.49
  97.00%    1.881         108.04         308.04
  98.00%    2.054         117.98         317.98
  99.00%    2.326         133.64         333.64


## Student Exercise Ideas

1. Keep mean demand fixed and double `stdev_demand_per_period`. What changes most?
2. Compare a deterministic lead time (`stdev_lead_time_periods = 0`) vs uncertain lead time.
3. For fill rate, test how changing `order_quantity` affects required safety stock.
4. Add a periodic review (`review_period_periods > 0`) and discuss the impact on protection period.